# ECOM6003 Project 2 — Univariate State Final Demand Forecasting

This notebook implements the univariate forecasting component of the State Final Demand (SFD) project. It estimates and evaluates Random Walk with Drift and ARIMA models for the six Australian states included in the dataset.

### Analysis included

- Random Walk with Drift benchmark for each state
- ARIMA model estimation and specification selection
- Fixed training/test split with a four-quarter out-of-sample backtest
- Forecast accuracy comparison using RMSE and MAE
- Re-estimation on the full sample and four-quarter operational forecasts
- Export of tables and figures for the report and later cross-model comparison

### Data and evaluation setup

- States: **NSW, VIC, QLD, SA, WA, TAS**
- Series: quarterly, seasonally adjusted, chain volume measures of SFD in levels
- Full sample: **1985 Q3–2026 Q2 (164 observations)**
- Training sample: **1985 Q3–2025 Q2 (160 observations)**
- Test sample: **2025 Q3–2026 Q2 (4 observations)**
- Operational forecast horizon: **2026 Q3–2027 Q2 (4 quarters)**
- Accuracy measures: **RMSE and MAE**

Earlier stationarity testing found that all six SFD level series were non-stationary and became stationary after first differencing. They are therefore treated as **I(1)** here, and the ARIMA candidate models use **d = 1**.

The four-quarter holdout period is used only for forecast evaluation. ARIMA specification selection is based on the training sample.


## 1. Setup

Import the required packages and create the output directory. The notebook can be run from top to bottom in Google Colab or a compatible Jupyter environment.


In [ ]:
# Core packages
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

# Reproducible display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

OUTPUT_DIR = Path('/content/ECOM6003_Jacky_outputs') if Path('/content').exists() else Path('./ECOM6003_Jacky_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Output folder:', OUTPUT_DIR.resolve())


## 2. Load and prepare `SFD_clean.xlsx`

The workbook is read from the working directory or standard Colab location. If the file is not found in Colab, the notebook prompts for an upload.


In [ ]:
# Locate or upload the workbook
candidate_paths = [
    Path('/content/SFD_clean.xlsx'),
    Path('/mnt/data/SFD_clean.xlsx'),
    Path('SFD_clean.xlsx'),
]

DATA_FILE = next((p for p in candidate_paths if p.exists()), None)

if DATA_FILE is None:
    try:
        from google.colab import files
        print('Please upload SFD_clean.xlsx')
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError('No file uploaded.')
        uploaded_name = next(iter(uploaded.keys()))
        DATA_FILE = Path('/content') / uploaded_name
    except ImportError:
        raise FileNotFoundError('SFD_clean.xlsx was not found. Put it beside the notebook and rerun this cell.')

print('Using data file:', DATA_FILE)

# Read the first worksheet
raw = pd.read_excel(DATA_FILE, sheet_name=0)
raw.columns = [str(c).strip() for c in raw.columns]

STATE_COLS = ['NSW', 'VIC', 'QLD', 'SA', 'WA', 'TAS']
required_cols = ['Quarter'] + STATE_COLS
missing = [c for c in required_cols if c not in raw.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Robust handling of Excel serial dates vs. already-parsed dates
q = raw['Quarter']
if pd.api.types.is_numeric_dtype(q):
    q_date = pd.to_datetime(q, unit='D', origin='1899-12-30')
else:
    q_date = pd.to_datetime(q)

sfd = raw[required_cols].copy()
sfd['Quarter'] = q_date.dt.to_period('Q')
sfd = sfd.sort_values('Quarter').reset_index(drop=True)

# Ensure numeric state series
for c in STATE_COLS:
    sfd[c] = pd.to_numeric(sfd[c], errors='raise')

print(f'Rows: {len(sfd)}')
print('Sample:', sfd['Quarter'].iloc[0], 'to', sfd['Quarter'].iloc[-1])
display(sfd.head())
display(sfd.tail())


## 3. Define the training and test samples

The analysis uses the following fixed split:

- Training sample ends at **2025 Q2**
- Test sample contains **2025 Q3, 2025 Q4, 2026 Q1 and 2026 Q2**

The same holdout period is used for both univariate approaches so that forecast accuracy is directly comparable.


In [ ]:
TRAIN_END = pd.Period('2025Q2', freq='Q')
TEST_START = pd.Period('2025Q3', freq='Q')
TEST_END = pd.Period('2026Q2', freq='Q')

train = sfd[sfd['Quarter'] <= TRAIN_END].copy()
test = sfd[(sfd['Quarter'] >= TEST_START) & (sfd['Quarter'] <= TEST_END)].copy()

print('Training observations:', len(train), '|', train['Quarter'].iloc[0], 'to', train['Quarter'].iloc[-1])
print('Test observations:', len(test), '|', test['Quarter'].iloc[0], 'to', test['Quarter'].iloc[-1])

if len(train) != 160:
    print('WARNING: expected 160 training observations.')
if len(test) != 4:
    raise ValueError('The test period must contain exactly 4 quarterly observations.')
if len(sfd) != 164:
    print('WARNING: expected 164 observations in the full dataset.')

split_summary = pd.DataFrame({
    'Dataset': ['Training', 'Test', 'Full dataset', 'Operational forecast'],
    'Period': ['1985 Q3–2025 Q2', '2025 Q3–2026 Q2', '1985 Q3–2026 Q2', '2026 Q3–2027 Q2'],
    'Observations': [len(train), len(test), len(sfd), 4],
    'Purpose': [
        'Estimate and select models',
        'Out-of-sample 4-quarter forecast evaluation',
        'Re-estimate selected specifications',
        'Future 4-quarter forecast'
    ]
})
display(split_summary)
split_summary.to_csv(OUTPUT_DIR / '01_sample_split.csv', index=False)


## 4. Random Walk with Drift benchmark

For each state, the benchmark model is:

\[
y_t = y_{t-1} + c + \varepsilon_t
\]

where \(c\) is the average quarterly change estimated from the training sample. The \(h\)-step forecast is:

\[
\hat y_{T+h} = y_T + h\hat c
\]


In [ ]:
def random_walk_with_drift_forecast(y_train, h=4):
    y_train = np.asarray(y_train, dtype=float)
    drift = np.mean(np.diff(y_train))
    forecast = y_train[-1] + drift * np.arange(1, h + 1)
    return forecast, float(drift)

rwd_rows = []
rwd_forecasts = {}

for state in STATE_COLS:
    fc, drift = random_walk_with_drift_forecast(train[state].values, h=4)
    rwd_forecasts[state] = fc
    rwd_rows.append({'State': state, 'Estimated_quarterly_drift': drift})

rwd_summary = pd.DataFrame(rwd_rows)
display(rwd_summary)
rwd_summary.to_csv(OUTPUT_DIR / '02_random_walk_drift_parameters.csv', index=False)


## 5. ARIMA specification search and selection

All ARIMA candidates are estimated using the training sample only. The search considers low-order ARIMA(p,1,q) specifications with \(p,q \leq 3\), both with and without drift.

Residual diagnostics are considered first. Models with a Ljung–Box p-value above 0.05 are treated as acceptable. Among the passing candidates, the model with the lowest BIC is selected, with AIC and model complexity used as tie-breakers. If no low-order candidate passes the residual diagnostic, the lowest-BIC specification is retained as a benchmark and clearly flagged.

Holdout RMSE and MAE are not used to choose the ARIMA specification.


In [ ]:
# Configuration
MAX_P = 3
MAX_Q = 3
LB_LAG = 8
DIAGNOSTIC_ALPHA = 0.05

# Optional manual overrides after training-sample diagnostic review.
# Format: 'STATE': {'order': (p, 1, q), 'trend': 't' or 'n'}
MANUAL_OVERRIDE = {}


def standardized_innovations(fit):
    err = np.asarray(fit.filter_results.standardized_forecasts_error[0], dtype=float)
    burn = max(int(getattr(fit, 'loglikelihood_burn', 0)), 1)
    return err[burn:]


def ljung_box_pvalue(fit, lag=8):
    err = standardized_innovations(fit)
    if len(err) <= lag:
        return np.nan
    out = acorr_ljungbox(err, lags=[lag], return_df=True)
    return float(out['lb_pvalue'].iloc[0])


def fit_candidate_models(y, state):
    records = []
    fit_map = {}

    for p in range(MAX_P + 1):
        for q in range(MAX_Q + 1):
            for trend in ['n', 't']:
                order = (p, 1, q)
                key = (order, trend)
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        fit = ARIMA(y, order=order, trend=trend).fit()
                    lb_p = ljung_box_pvalue(fit, LB_LAG)
                    records.append({
                        'State': state,
                        'p': p,
                        'd': 1,
                        'q': q,
                        'Trend': trend,
                        'Drift': 'Yes' if trend == 't' else 'No',
                        'AIC': float(fit.aic),
                        'BIC': float(fit.bic),
                        f'Ljung_Box_p_lag{LB_LAG}': lb_p,
                        'Diagnostic_Pass': bool(lb_p >= DIAGNOSTIC_ALPHA) if not np.isnan(lb_p) else False,
                        'Fit_Status': 'OK'
                    })
                    fit_map[key] = fit
                except Exception as exc:
                    records.append({
                        'State': state,
                        'p': p,
                        'd': 1,
                        'q': q,
                        'Trend': trend,
                        'Drift': 'Yes' if trend == 't' else 'No',
                        'AIC': np.nan,
                        'BIC': np.nan,
                        f'Ljung_Box_p_lag{LB_LAG}': np.nan,
                        'Diagnostic_Pass': False,
                        'Fit_Status': f'FAILED: {type(exc).__name__}'
                    })

    cand = pd.DataFrame(records)
    ok = cand[cand['Fit_Status'] == 'OK'].copy()
    if ok.empty:
        raise RuntimeError(f'No ARIMA candidate could be fitted for {state}.')

    ok['Complexity_p_plus_q'] = ok['p'] + ok['q']
    ok['AIC_rank'] = ok['AIC'].rank(method='min')
    ok['BIC_rank'] = ok['BIC'].rank(method='min')
    ok['IC_average_rank'] = (ok['AIC_rank'] + ok['BIC_rank']) / 2

    if state in MANUAL_OVERRIDE:
        spec = MANUAL_OVERRIDE[state]
        order = tuple(spec['order'])
        trend = spec['trend']
        row = ok[(ok['p'] == order[0]) & (ok['d'] == order[1]) & (ok['q'] == order[2]) & (ok['Trend'] == trend)]
        if row.empty:
            raise ValueError(f'Manual override for {state} was not found among successful candidates.')
        selected = row.iloc[0]
        source = 'Manual override after diagnostic review'
    else:
        passing = ok[ok['Diagnostic_Pass']].copy()
        if not passing.empty:
            # Diagnostics first. BIC is the primary selection criterion because it
            # penalises unnecessary complexity; AIC and p+q are tie-breakers.
            selected = passing.sort_values(['BIC', 'AIC', 'Complexity_p_plus_q', 'p', 'q']).iloc[0]
            source = 'Ljung-Box pass first; lowest BIC among passing candidates'
        else:
            # If no candidate passes the diagnostic, retain the lowest-BIC
            # low-order specification as a benchmark and flag the issue.
            selected = ok.sort_values(['BIC', 'AIC', 'Complexity_p_plus_q', 'p', 'q']).iloc[0]
            source = 'No low-order candidate passed Ljung-Box; lowest-BIC fallback (flagged)'

    order = (int(selected['p']), 1, int(selected['q']))
    trend = selected['Trend']
    fit = fit_map[(order, trend)]

    ok = ok.sort_values(['Diagnostic_Pass', 'BIC', 'AIC', 'Complexity_p_plus_q'], ascending=[False, True, True, True]).reset_index(drop=True)
    return ok, selected, fit, source


candidate_tables = {}
selected_rows = []
selected_fits = {}
selected_specs = {}

for state in STATE_COLS:
    candidates, selected, fit, source = fit_candidate_models(train[state].values, state)
    candidate_tables[state] = candidates
    selected_fits[state] = fit
    selected_specs[state] = {
        'order': (int(selected['p']), 1, int(selected['q'])),
        'trend': selected['Trend']
    }

    lb_p = float(selected[f'Ljung_Box_p_lag{LB_LAG}'])
    selected_rows.append({
        'State': state,
        'Selected_Model': f"ARIMA({int(selected['p'])},1,{int(selected['q'])})" + (' + drift' if selected['Trend'] == 't' else ''),
        'AIC': float(selected['AIC']),
        'BIC': float(selected['BIC']),
        f'Ljung_Box_p_lag{LB_LAG}': lb_p,
        'Residual_Diagnostic': 'PASS' if lb_p >= DIAGNOSTIC_ALPHA else 'FLAG: residual autocorrelation remains',
        'Selection_Source': source
    })

    candidates.to_csv(OUTPUT_DIR / f'03_ARIMA_candidates_{state}.csv', index=False)

selected_models = pd.DataFrame(selected_rows)
print('SELECTED ARIMA SPECIFICATIONS')
display(selected_models)
selected_models.to_csv(OUTPUT_DIR / '03_selected_ARIMA_models.csv', index=False)

for state in STATE_COLS:
    print('\n', '=' * 18, state, '=' * 18)
    display(candidate_tables[state].head(10))


### Residual diagnostic note

A Ljung–Box p-value at or below 0.05 indicates that statistically detectable serial correlation remains in the residuals. Any selected specification with this result is flagged and should be interpreted cautiously.

`MANUAL_OVERRIDE` is available only if a different specification is justified using training-sample diagnostics. The holdout period should not be used for specification selection.


## 6. Four-quarter out-of-sample backtest

Both univariate approaches are estimated using observations through **2025 Q2** and then used to produce a single four-quarter forecast path for **2025 Q3–2026 Q2**. The realised test values are used only to calculate forecast errors.


In [ ]:
def rmse(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    return float(np.sqrt(np.mean((actual - forecast) ** 2)))


def mae(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    return float(np.mean(np.abs(actual - forecast)))


backtest_rows = []
accuracy_rows = []

for state in STATE_COLS:
    actual = test[state].to_numpy(dtype=float)

    # Random Walk with Drift
    rwd_fc = np.asarray(rwd_forecasts[state], dtype=float)

    # Selected ARIMA
    arima_fc = np.asarray(selected_fits[state].forecast(steps=4), dtype=float)
    arima_name = selected_models.loc[selected_models['State'] == state, 'Selected_Model'].iloc[0]

    for i, quarter in enumerate(test['Quarter']):
        backtest_rows.append({
            'Quarter': str(quarter),
            'State': state,
            'Actual': actual[i],
            'Random_Walk_with_Drift': rwd_fc[i],
            'ARIMA_Forecast': arima_fc[i],
            'ARIMA_Model': arima_name
        })

    accuracy_rows.extend([
        {'State': state, 'Model': 'Random Walk with Drift', 'RMSE': rmse(actual, rwd_fc), 'MAE': mae(actual, rwd_fc)},
        {'State': state, 'Model': arima_name, 'RMSE': rmse(actual, arima_fc), 'MAE': mae(actual, arima_fc)},
    ])

backtest_forecasts = pd.DataFrame(backtest_rows)
univariate_accuracy = pd.DataFrame(accuracy_rows)

print('BACKTEST FORECASTS')
display(backtest_forecasts)
print('\nUNIVARIATE ACCURACY (lower is better)')
display(univariate_accuracy.sort_values(['State', 'RMSE']))

backtest_forecasts.to_csv(OUTPUT_DIR / '04_backtest_forecasts_2025Q3_2026Q2.csv', index=False)
univariate_accuracy.to_csv(OUTPUT_DIR / '04_univariate_accuracy_RMSE_MAE.csv', index=False)


## 7. Compare univariate forecast accuracy

The table compares Random Walk with Drift with the selected ARIMA specification for each state. RMSE and MAE are reported separately, with lower values indicating better forecast accuracy.

Where the differences are extremely small, they should be treated as practically equivalent rather than as evidence that one model clearly outperforms the other. This is particularly relevant when the selected model is ARIMA(0,1,0) with drift, which is closely related to the Random Walk with Drift benchmark.


In [ ]:
comparison_rows = []
for state in STATE_COLS:
    block = univariate_accuracy[univariate_accuracy['State'] == state].copy()
    rmse_best = block.loc[block['RMSE'].idxmin(), 'Model']
    mae_best = block.loc[block['MAE'].idxmin(), 'Model']
    overall = rmse_best if rmse_best == mae_best else f'Mixed evidence: RMSE={rmse_best}; MAE={mae_best}'

    rw = block[block['Model'] == 'Random Walk with Drift'].iloc[0]
    ar = block[block['Model'] != 'Random Walk with Drift'].iloc[0]

    comparison_rows.append({
        'State': state,
        'RWD_RMSE': rw['RMSE'],
        'RWD_MAE': rw['MAE'],
        'ARIMA_Model': ar['Model'],
        'ARIMA_RMSE': ar['RMSE'],
        'ARIMA_MAE': ar['MAE'],
        'Best_by_RMSE': rmse_best,
        'Best_by_MAE': mae_best,
        'Univariate_conclusion': overall
    })

univariate_comparison = pd.DataFrame(comparison_rows)
display(univariate_comparison)
univariate_comparison.to_csv(OUTPUT_DIR / '05_univariate_model_comparison_by_state.csv', index=False)


## 8. Backtest plots

Each plot shows the final 20 training quarters and the four-quarter test period. The two forecast paths are plotted against the realised SFD values in the holdout sample.


In [ ]:
plot_start = max(0, len(train) - 20)

for state in STATE_COLS:
    history_q = train['Quarter'].iloc[plot_start:].astype(str).tolist()
    history_y = train[state].iloc[plot_start:].to_numpy(dtype=float)
    test_q = test['Quarter'].astype(str).tolist()
    actual = test[state].to_numpy(dtype=float)

    block = backtest_forecasts[backtest_forecasts['State'] == state]
    rwd_fc = block['Random_Walk_with_Drift'].to_numpy(dtype=float)
    ar_fc = block['ARIMA_Forecast'].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(history_q, history_y, marker='o', label='Training actual')
    ax.plot(test_q, actual, marker='o', label='Test actual')
    ax.plot(test_q, rwd_fc, marker='o', linestyle='--', label='Random Walk with Drift')
    ax.plot(test_q, ar_fc, marker='o', linestyle='--', label='Selected ARIMA')
    ax.set_title(f'{state}: 4-Quarter Out-of-Sample Backtest')
    ax.set_xlabel('Quarter')
    ax.set_ylabel('State Final Demand')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'05_backtest_{state}.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 9. Refit selected specifications and produce operational forecasts

After the backtest, the previously selected ARIMA specification for each state is re-estimated using the full sample through **2026 Q2**. The specification itself is not re-selected using the test results.

The notebook then produces four-quarter forecasts for **2026 Q3–2027 Q2**. Random Walk with Drift forecasts are also retained so that either univariate approach can be used later if selected by the final cross-model comparison.


In [ ]:
last_q = sfd['Quarter'].iloc[-1]
future_quarters = [last_q + i for i in range(1, 5)]

operational_rows = []
full_arima_fits = {}

for state in STATE_COLS:
    y_full = sfd[state].to_numpy(dtype=float)

    # Random Walk with Drift, re-estimated on full data
    rwd_fc, full_drift = random_walk_with_drift_forecast(y_full, h=4)

    # Refit the selected ARIMA specification on all 164 observations
    spec = selected_specs[state]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        full_fit = ARIMA(y_full, order=spec['order'], trend=spec['trend']).fit()
    full_arima_fits[state] = full_fit

    pred = full_fit.get_forecast(steps=4)
    sf = pred.summary_frame(alpha=0.05)
    arima_mean = np.asarray(sf['mean'], dtype=float)
    lower = np.asarray(sf['mean_ci_lower'], dtype=float)
    upper = np.asarray(sf['mean_ci_upper'], dtype=float)
    arima_name = selected_models.loc[selected_models['State'] == state, 'Selected_Model'].iloc[0]

    for i, qtr in enumerate(future_quarters):
        operational_rows.append({
            'Quarter': str(qtr),
            'State': state,
            'Model': 'Random Walk with Drift',
            'Forecast': float(rwd_fc[i]),
            'Lower_95': np.nan,
            'Upper_95': np.nan
        })
        operational_rows.append({
            'Quarter': str(qtr),
            'State': state,
            'Model': arima_name,
            'Forecast': float(arima_mean[i]),
            'Lower_95': float(lower[i]),
            'Upper_95': float(upper[i])
        })

operational_forecasts = pd.DataFrame(operational_rows)
print('UNIVARIATE OPERATIONAL FORECASTS: 2026 Q3–2027 Q2')
display(operational_forecasts)
operational_forecasts.to_csv(OUTPUT_DIR / '06_operational_forecasts_univariate_2026Q3_2027Q2.csv', index=False)


## 10. Operational forecast plots

These figures show the four-quarter forecasts for **2026 Q3–2027 Q2** together with recent observed SFD values. ARIMA forecasts include 95% prediction intervals.


In [ ]:
plot_start = max(0, len(sfd) - 16)

for state in STATE_COLS:
    hist_q = sfd['Quarter'].iloc[plot_start:].astype(str).tolist()
    hist_y = sfd[state].iloc[plot_start:].to_numpy(dtype=float)

    block = operational_forecasts[operational_forecasts['State'] == state]
    rwd = block[block['Model'] == 'Random Walk with Drift'].sort_values('Quarter')
    ar = block[block['Model'] != 'Random Walk with Drift'].sort_values('Quarter')

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(hist_q, hist_y, marker='o', label='Actual SFD')
    ax.plot(rwd['Quarter'], rwd['Forecast'], marker='o', linestyle='--', label='Random Walk with Drift forecast')
    ax.plot(ar['Quarter'], ar['Forecast'], marker='o', linestyle='--', label='Selected ARIMA forecast')
    ax.fill_between(ar['Quarter'], ar['Lower_95'], ar['Upper_95'], alpha=0.2, label='ARIMA 95% interval')
    ax.set_title(f'{state}: Operational Forecast, 2026 Q3–2027 Q2')
    ax.set_xlabel('Quarter')
    ax.set_ylabel('State Final Demand')
    ax.tick_params(axis='x', rotation=45)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'06_operational_forecast_{state}.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 11. Optional cross-model comparison with VAR/VECM results

Once multivariate backtest results are available, save them as `multivariate_accuracy.csv` with the following columns:

| State | Model | RMSE | MAE |
|---|---|---:|---:|
| NSW | VECM | ... | ... |
| VIC | VECM | ... | ... |
| ... | ... | ... | ... |

The optional cell below combines the multivariate results with the univariate accuracy table. If RMSE and MAE identify different winners, the comparison is reported as mixed rather than forcing a single conclusion.


In [ ]:
# Create a blank template for multivariate accuracy results
multivariate_template = pd.DataFrame({
    'State': STATE_COLS,
    'Model': ['VAR/VECM'] * len(STATE_COLS),
    'RMSE': [np.nan] * len(STATE_COLS),
    'MAE': [np.nan] * len(STATE_COLS),
})
multivariate_template.to_csv(OUTPUT_DIR / '07_multivariate_accuracy_TEMPLATE.csv', index=False)
display(multivariate_template)

# Optional automatic merge if the completed CSV is present
possible_files = [
    Path('/content/multivariate_accuracy.csv'),
    Path('/mnt/data/multivariate_accuracy.csv'),
    Path('multivariate_accuracy.csv'),
]
multivariate_file = next((p for p in possible_files if p.exists()), None)

if multivariate_file is None:
    print('Multivariate accuracy file not found. The template has been created for later use.')
    print('Template saved to:', OUTPUT_DIR / '07_multivariate_accuracy_TEMPLATE.csv')
else:
    multi = pd.read_csv(multivariate_file)
    required = {'State', 'Model', 'RMSE', 'MAE'}
    if not required.issubset(multi.columns):
        raise ValueError(f'Multivariate accuracy CSV must contain columns: {sorted(required)}')

    combined = pd.concat([univariate_accuracy, multi[['State', 'Model', 'RMSE', 'MAE']]], ignore_index=True)
    combined = combined.sort_values(['State', 'RMSE']).reset_index(drop=True)
    display(combined)
    combined.to_csv(OUTPUT_DIR / '07_combined_accuracy_all_models.csv', index=False)

    decisions = []
    for state in STATE_COLS:
        block = combined[combined['State'] == state]
        rmse_best = block.loc[block['RMSE'].idxmin(), 'Model']
        mae_best = block.loc[block['MAE'].idxmin(), 'Model']
        conclusion = rmse_best if rmse_best == mae_best else f'Mixed: RMSE={rmse_best}; MAE={mae_best}'
        decisions.append({'State': state, 'Best_by_RMSE': rmse_best, 'Best_by_MAE': mae_best, 'Conclusion': conclusion})

    final_comparison = pd.DataFrame(decisions)
    print('CROSS-MODEL COMPARISON')
    display(final_comparison)
    final_comparison.to_csv(OUTPUT_DIR / '07_model_comparison_decisions.csv', index=False)


## 12. Export results

The notebook writes the main tables, figures and a short text summary to the output directory, then creates a ZIP archive. Backtest results and future operational forecasts are kept separate.


In [ ]:
# Compact text summary
summary_lines = []
summary_lines.append('ECOM6003 — Univariate SFD Forecasting Summary')
summary_lines.append('Training: 1985 Q3–2025 Q2')
summary_lines.append('Test: 2025 Q3–2026 Q2')
summary_lines.append('Operational forecast: 2026 Q3–2027 Q2')
summary_lines.append('Metrics: RMSE, MAE')
summary_lines.append('')
summary_lines.append('Selected ARIMA specifications (diagnostics-first selection):')
for _, r in selected_models.iterrows():
    summary_lines.append(
        f"{r['State']}: {r['Selected_Model']}; AIC={r['AIC']:.2f}; BIC={r['BIC']:.2f}; "
        f"LB p={r[f'Ljung_Box_p_lag{LB_LAG}']:.4f}; {r['Residual_Diagnostic']}"
    )
summary_lines.append('')
summary_lines.append('Univariate backtest comparison:')
for _, r in univariate_comparison.iterrows():
    summary_lines.append(f"{r['State']}: {r['Univariate_conclusion']}")

summary_text = '\n'.join(summary_lines)
print(summary_text)
(OUTPUT_DIR / '00_README_results_summary.txt').write_text(summary_text, encoding='utf-8')

# Zip outputs
zip_base = str(OUTPUT_DIR)
zip_path = shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
print('\nCreated output ZIP:', zip_path)
print('\nFiles created:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print('-', p.name)

# In Google Colab, uncomment the next two lines if you want an immediate browser download:
# from google.colab import files
# files.download(zip_path)
